# SQL Business Analysis

**Import Libraries&Load datasets**

In [3]:
import sqlite3
import pandas as pd

In [4]:
transactions = pd.read_csv('transactions')
customers = pd.read_csv('customers')

In [6]:
connect = sqlite3.connect(":memory:")

In [9]:
transactions.to_sql("transactions", connect, index=False)
customers.to_sql("customers", connect, index=False)

1500

## 1. How many customers are in each customer segment?

In [15]:
pd.read_sql_query(
    """
    SELECT
        Customer_Segment,
        COUNT(*) AS Customer_Count
    FROM customers
    GROUP BY Customer_Segment
    ORDER BY Customer_Count DESC;
    """,
    connect
)

,Customer_Segment,Customer_Count
0,Everyday,711
1,Premium,295
2,Student,278
3,SME,216


**Result:**
The Everyday segment has the largest number of customers with 711 customers, followed by Premium with 295, Student with 278, and SME with 216.

**Business Interpretation:**
FinTrust's customer base is mainly made up of Everyday customers. This segment represents the largest customer group and is therefore important when analysing overall customer behaviour and transaction activity.

## 2. Which transaction types have the highest transaction activity?

In [14]:
pd.read_sql_query(
    """
    SELECT
        Transaction_Type,
        COUNT(*) AS Transaction_Count
    FROM transactions
    GROUP BY Transaction_Type
    ORDER BY Transaction_Count DESC;
    """,
    connect
)

,Transaction_Type,Transaction_Count
0,Transfer,3549
1,Card Purchase,3033
2,Bill Payment,1475
3,Cash Withdrawal,1430
4,Deposit,1328
5,Airtime/Data,1185


**Result:**
Transfers had the highest transaction volume with 3,549 transactions, followed by Card Purchases with 3,033. Airtime/Data had the lowest volume with 1,185 transactions.

**Business Interpretation:**
Transfers and Card Purchases account for a large share of transaction activity, showing that these are important services within the FinTrust transaction ecosystem. They may therefore require particular attention when analysing customer usage and transaction performance.

## 3. Which transaction types generate the highest transaction value?

In [16]:
pd.read_sql_query(
    """
    SELECT
        Transaction_Type,
        AVG(Amount_NGN) AS Avg_Transaction_Value,
        SUM(Amount_NGN) AS Total_Transaction_Value
    FROM transactions
    GROUP BY Transaction_Type
    ORDER BY Total_Transaction_Value DESC;
    """,
    connect
)

,Transaction_Type,Avg_Transaction_Value,Total_Transaction_Value
0,Transfer,67037.672699,2.379167e+08
1,Deposit,98633.328200,1.309851e+08
2,Card Purchase,28309.730887,8.586341e+07
3,Cash Withdrawal,47393.260217,6.777236e+07
4,Bill Payment,19093.998312,2.816365e+07
5,Airtime/Data,8249.933494,9.776171e+06


**Result:**
Transfers generated the highest total transaction value at approximately NGN 237.92 million, followed by Deposits at approximately NGN 130.99 million. Transfers also had the highest average transaction value at approximately NGN 67,037.67, while Deposits had an average of approximately NGN 98,633.33, which was actually the highest average transaction value among the transaction types.

**Business Interpretation:**
Transfers contribute the most to overall transaction value because of their high transaction volume, while Deposits have the highest average transaction value. This shows that transaction volume and transaction value do not always follow the same pattern.

## 4. Which channels are used most frequently?

In [18]:
pd.read_sql_query(
    """
    SELECT
        Channel,
        COUNT(*) AS Total_Transactions,
        SUM(Amount_NGN) AS Total_Transaction_Value
    FROM transactions
    GROUP BY Channel
    ORDER BY Total_Transaction_Value DESC;
    """,
    connect
)

,Channel,Total_Transactions,Total_Transaction_Value
0,Mobile App,5102,2.401044e+08
1,POS,2393,1.043467e+08
2,Web,1869,8.932500e+07
3,ATM,1747,8.394235e+07
4,USSD,889,4.275895e+07


**Result:**
The Mobile App recorded the highest transaction activity with 5,102 transactions and the highest total transaction value at approximately NGN 240.10 million. USSD recorded the lowest activity with 889 transactions and approximately NGN 42.76 million in transaction value.

**Business Interpretation:**
The Mobile App is the most active transaction channel in the dataset, both in terms of transaction volume and value. This indicates that digital app activity is an important part of FinTrust's customer transaction behaviour.

## 5. What is the transaction success rate?

In [22]:
pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS Total_Transactions,
        SUM(
            CASE
                WHEN Transaction_Status = 'Successful' THEN 1
                ELSE 0
            END
        ) AS Successful_Transactions,
        ROUND(
            100.0 * SUM(
                CASE
                    WHEN Transaction_Status = 'Successful' THEN 1
                    ELSE 0
                END
                ) / COUNT(*),
                2
        ) AS Success_Rate
    FROM transactions;
    """,
    connect
)

,Total_Transactions,Successful_Transactions,Success_Rate
0,12000,10856,90.47


**Result:**
Out of 12,000 transactions, 10,856 were successful, giving an overall transaction success rate of 90.47%.

**Business Interpretation:**
The majority of transactions were completed successfully. However, approximately 9.53% of transactions were not successful, which provides an area for further investigation into transaction failures and their possible causes.

## 6. How does transaction activity differ across customer segments?

In [24]:
pd.read_sql_query(
    """
    SELECT
        c.Customer_Segment,
        COUNT(t.Transaction_ID) AS Total_Transactions,
        SUM(t.Amount_NGN) AS Total_Transaction_Value
    FROM transactions t
    JOIN customers c
        ON c.Customer_ID = t.Customer_ID
    GROUP BY c.Customer_Segment
    ORDER BY Total_Transaction_Value DESC;
    """,
    connect
)

,Customer_Segment,Total_Transactions,Total_Transaction_Value
0,Everyday,5644,2.614589e+08
1,Premium,2361,1.077512e+08
2,Student,2289,1.074759e+08
3,SME,1706,8.379137e+07


**Result:**
Everyday customers generated the highest transaction activity with 5,644 transactions and the highest total transaction value at approximately NGN 261.46 million. SME customers had the lowest transaction activity with 1,706 transactions and approximately NGN 83.79 million in transaction value.

**Business Interpretation:**
The Everyday segment contributes the largest share of transaction activity and value in the dataset. This is partly consistent with it being the largest customer segment, so further analysis of transaction behaviour per customer could provide a clearer comparison between segments.

## 7. Is risk review more common for international transactions?

In [27]:
pd.read_sql_query(
    """
    SELECT
        International_Transaction,
        COUNT(*) AS Total_Transactions,
        SUM(
            CASE
                WHEN Risk_Review_Flag = 'Yes' THEN 1
                ELSE 0
            END
        ) AS Risk_Review_Count,
        ROUND(
            100.0 * SUM(
                CASE
                    WHEN Risk_Review_Flag = 'Yes' THEN 1
                    ELSE 0
                END) / COUNT(*),
                2
            ) AS Risk_Review_Rate
    FROM transactions
    GROUP BY International_Transaction
    ORDER BY Risk_Review_Rate DESC;
    """,
    connect
)

,International_Transaction,Total_Transactions,Risk_Review_Count,Risk_Review_Rate
0,Yes,480,177,36.88
1,No,11520,2175,18.88


**Result:**
International transactions had a risk review rate of 36.88%, compared with 18.88% for non international transactions. There were 177 risk reviewed international transactions out of 480, compared with 2,175 out of 11,520 non international transactions.

**Business Interpretation:**
International transactions had a substantially higher risk review rate than non international transactions in this dataset. This suggests that international transaction status may be an important factor to investigate further when analysing risk review patterns.

## 8. Which transaction channels have the highest risk review rate?

In [28]:
pd.read_sql_query(
    """
    SELECT
        Channel,
        COUNT(*) AS Total_Transactions,
        SUM(
            CASE
                WHEN Risk_Review_Flag = 'Yes' THEN 1
                ELSE 0
            END
        ) AS Risk_Review_Count,
        ROUND(
            100.0 * SUM(
                CASE
                    WHEN Risk_Review_Flag = 'Yes' THEN 1
                    ELSE 0
                END) / COUNT(*),
                2
            ) AS Risk_Review_Rate
    FROM transactions
    GROUP BY Channel
    ORDER BY Risk_Review_Rate DESC;
    """,
    connect
)

,Channel,Total_Transactions,Risk_Review_Count,Risk_Review_Rate
0,Web,1869,399,21.35
1,ATM,1747,370,21.18
2,Mobile App,5102,990,19.40
3,POS,2393,439,18.35
4,USSD,889,154,17.32


**Result:**
Web transactions had the highest risk review rate at 21.35%, followed closely by ATM transactions at 21.18%. USSD had the lowest rate at 17.32%. Mobile App had a 19.40% risk review rate, while POS had 18.35%.

**Business Interpretation:**
Risk review rates differ across transaction channels. Web and ATM transactions had the highest rates in the dataset, while USSD had the lowest. This suggests that transaction channel could be useful when investigating factors associated with risk reviews.